In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 1. Load DataFrame
df = pd.read_csv('C:\\Users\\liad1\\OneDrive\\מסמכים\\personal git\\work_assignments\\cyber_1\\intelos_task_detect_multi_device_user.csv',
                 parse_dates=['REQUEST_TIME'], dayfirst=False)
df.head()

,REQUEST_TIME,DEVICE_IP,ID,USER_ID,ID_TYPE,OPERATING_SYSTEM
0,2024-10-01 09:17:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android
1,2024-10-01 09:19:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android
2,2024-10-01 09:22:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android
3,2024-10-01 09:27:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android
4,2024-10-01 09:36:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android


In [ ]:
# df["REQUEST_TIME"] = pd.to_datetime(
#     df["REQUEST_TIME"],
#     format="%Y/%m/%d %H:%M",
#     errors="coerce"
# )

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7915 entries, 0 to 7914
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   REQUEST_TIME      7915 non-null   datetime64[us]
 1   DEVICE_IP         7915 non-null   str           
 2   ID                7915 non-null   str           
 3   USER_ID           7915 non-null   int64         
 4   ID_TYPE           7915 non-null   str           
 5   OPERATING_SYSTEM  7915 non-null   str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 371.1 KB


In [4]:
# 2. overall unique count Aggregation by USER_ID:
user_summary = df.groupby('USER_ID').agg(
    unique_os=('OPERATING_SYSTEM', 'nunique'),
    unique_id_type=('ID_TYPE', 'nunique'),
    unique_id=('ID', 'nunique'),
    unique_ip=('DEVICE_IP', 'nunique'),
    user_total_reqs =('USER_ID', 'size')
).reset_index()
user_summary

,USER_ID,unique_os,unique_id_type,unique_id,unique_ip,user_total_reqs
0,1,1,1,3,54,1535
1,2,1,1,5,60,619
2,3,1,1,4,22,105
3,4,1,1,1,4,5
4,5,1,1,2,10,23
5,6,2,1,2,50,862
6,7,1,1,3,65,4766


In [5]:
#4. first\last seen and number of requests per device (ID) and user id
device_summary = df.groupby(['USER_ID', 'ID']).agg(
    first_seen=('REQUEST_TIME', 'min'),
    last_seen=('REQUEST_TIME', 'max'),
    device_requests=('ID', 'count')
).reset_index()
device_summary

,USER_ID,ID,first_seen,last_seen,device_requests
0,1,59f377d4-2056-4a6a-a18a-916c0f59445a,2024-10-30 13:29:00,2024-11-01 10:43:00,87
1,1,8819aef2-1a40-4291-b4a4-bcc08c8a4b74,2024-11-01 10:54:00,2024-11-29 14:38:00,1071
2,1,944e0661-f7f4-4bca-a831-f9f2231a3575,2024-10-01 10:37:00,2024-10-30 10:48:00,377
3,2,04de0e1c-de2f-4515-8423-ae3083e4acee,2024-10-02 09:39:00,2024-11-07 20:02:00,44
4,2,1a6196f1-7953-4e5a-8449-780be167de51,2024-10-03 08:59:00,2024-11-28 16:46:00,450
5,2,686931a4-abd8-46e6-b0e3-78e57c0c620c,2024-10-03 12:53:00,2024-11-28 13:58:00,102
6,2,7c833208-053f-4ef5-bd69-f17d11084b24,2024-11-10 16:59:00,2024-11-14 10:39:00,18
7,2,95d9aeb3-c446-4b44-b165-8f617e14955e,2024-10-12 11:29:00,2024-11-24 10:55:00,5
8,3,AABvrk7RhqwAABuvMhFb-w,2024-10-08 19:55:00,2024-11-16 12:11:00,24
9,3,AAFK807KLUoAABmn5BDNiQ,2024-10-05 09:39:00,2024-11-29 16:59:00,57


In [6]:
device_summary = device_summary.merge(user_summary[['USER_ID','user_total_reqs']], on='USER_ID', how='left')
device_summary.head()

,USER_ID,ID,first_seen,last_seen,device_requests,user_total_reqs
0,1,59f377d4-2056-4a6a-a18a-916c0f59445a,2024-10-30 13:29:00,2024-11-01 10:43:00,87,1535
1,1,8819aef2-1a40-4291-b4a4-bcc08c8a4b74,2024-11-01 10:54:00,2024-11-29 14:38:00,1071,1535
2,1,944e0661-f7f4-4bca-a831-f9f2231a3575,2024-10-01 10:37:00,2024-10-30 10:48:00,377,1535
3,2,04de0e1c-de2f-4515-8423-ae3083e4acee,2024-10-02 09:39:00,2024-11-07 20:02:00,44,619
4,2,1a6196f1-7953-4e5a-8449-780be167de51,2024-10-03 08:59:00,2024-11-28 16:46:00,450,619


In [7]:
#5. Calculate the measures for trhesholding:
device_summary['request_share'] = device_summary['device_requests'] / device_summary['user_total_reqs']
device_summary['activity_duration'] = device_summary['last_seen'] - device_summary['first_seen']
device_summary.head()

,USER_ID,ID,first_seen,last_seen,device_requests,user_total_reqs,request_share,activity_duration
0,1,59f377d4-2056-4a6a-a18a-916c0f59445a,2024-10-30 13:29:00,2024-11-01 10:43:00,87,1535,0.056678,1 days 21:14:00
1,1,8819aef2-1a40-4291-b4a4-bcc08c8a4b74,2024-11-01 10:54:00,2024-11-29 14:38:00,1071,1535,0.697720,28 days 03:44:00
2,1,944e0661-f7f4-4bca-a831-f9f2231a3575,2024-10-01 10:37:00,2024-10-30 10:48:00,377,1535,0.245603,29 days 00:11:00
3,2,04de0e1c-de2f-4515-8423-ae3083e4acee,2024-10-02 09:39:00,2024-11-07 20:02:00,44,619,0.071082,36 days 10:23:00
4,2,1a6196f1-7953-4e5a-8449-780be167de51,2024-10-03 08:59:00,2024-11-28 16:46:00,450,619,0.726979,56 days 07:47:00


In [8]:
# 5. filtering devices based on thresholds:
one_day = pd.Timedelta(days=1)
valid_devices = device_summary[
    (device_summary['request_share'] > 0.05) & 
    (device_summary['activity_duration'] > one_day)
].copy()
valid_devices.head()

,USER_ID,ID,first_seen,last_seen,device_requests,user_total_reqs,request_share,activity_duration
0,1,59f377d4-2056-4a6a-a18a-916c0f59445a,2024-10-30 13:29:00,2024-11-01 10:43:00,87,1535,0.056678,1 days 21:14:00
1,1,8819aef2-1a40-4291-b4a4-bcc08c8a4b74,2024-11-01 10:54:00,2024-11-29 14:38:00,1071,1535,0.697720,28 days 03:44:00
2,1,944e0661-f7f4-4bca-a831-f9f2231a3575,2024-10-01 10:37:00,2024-10-30 10:48:00,377,1535,0.245603,29 days 00:11:00
3,2,04de0e1c-de2f-4515-8423-ae3083e4acee,2024-10-02 09:39:00,2024-11-07 20:02:00,44,619,0.071082,36 days 10:23:00
4,2,1a6196f1-7953-4e5a-8449-780be167de51,2024-10-03 08:59:00,2024-11-28 16:46:00,450,619,0.726979,56 days 07:47:00


In [9]:
unvalid_devices = device_summary[
    (device_summary['request_share'] <= 0.05) |
    (device_summary['activity_duration'] <= one_day)
].copy()
unvalid_devices

,USER_ID,ID,first_seen,last_seen,device_requests,user_total_reqs,request_share,activity_duration
6,2,7c833208-053f-4ef5-bd69-f17d11084b24,2024-11-10 16:59:00,2024-11-14 10:39:00,18,619,0.029079,3 days 17:40:00
7,2,95d9aeb3-c446-4b44-b165-8f617e14955e,2024-10-12 11:29:00,2024-11-24 10:55:00,5,619,0.008078,42 days 23:26:00
17,7,AAA7PE7OD50AABQ_9xhR4Q,2024-10-01 09:17:00,2024-11-29 12:39:00,149,4766,0.031263,59 days 03:22:00


In [10]:
# 6. check timeline overlap of valid devices for each user:
def check_valid_device_overlap(group):
    # if there's only one or zero valid devices, we can directly classify as 0 (no multiple devices)
    if len(group) <= 1:
        return 0
    
    # sort devices by first_seen time
    sorted_group = group.sort_values('first_seen')
    
    #check for overlap in activity periods
    for i in range(len(sorted_group) - 1):
        current_last_seen = sorted_group.iloc[i]['last_seen']
        next_first_seen = sorted_group.iloc[i+1]['first_seen']
        
        if next_first_seen <= current_last_seen:
            return 1 #Found an overlap, classify as 1 (multiple devices)
            
    return 0 #There was a "replacement" in the device ID


In [11]:
#run function for each user group and create final classification DataFrame
user_classification = valid_devices.groupby('USER_ID').apply(check_valid_device_overlap).reset_index()
user_classification.columns = ['USER_ID', 'has_multiple_devices']
user_classification

,USER_ID,has_multiple_devices
0,1,0
1,2,1
2,3,1
3,4,0
4,5,1
5,6,1
6,7,1


In [ ]:
# 7. merege the classification back to the original DataFrame
final_df = df.merge(user_classification, on='USER_ID', how='left')
final_df['has_multiple_devices'] = final_df['has_multiple_devices'].fillna(0).astype(int)
final_df

,REQUEST_TIME,DEVICE_IP,ID,USER_ID,ID_TYPE,OPERATING_SYSTEM,has_multiple_devices
0,2024-10-01 09:17:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android,1
1,2024-10-01 09:19:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android,1
2,2024-10-01 09:22:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android,1
3,2024-10-01 09:27:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android,1
4,2024-10-01 09:36:00,2a00:5400:e266::,AAA7PE7OD50AABQ_9xhR4Q,7,WEB_ID,android,1
...,...,...,...,...,...,...,...
7910,2024-11-29 20:24:00,2a00:5400:e051:9f58::,AAFcvU7LIAoAABO-Xttw5A,7,WEB_ID,android,1
7911,2024-11-29 20:25:00,2a00:5400:e051:9f58::,AAFcvU7LIAoAABO-Xttw5A,7,WEB_ID,android,1
7912,2024-11-29 20:26:00,2a00:5400:e051:9f58::,AAFcvU7LIAoAABO-Xttw5A,7,WEB_ID,android,1
7913,2024-11-29 20:26:00,2a00:5400:e051:9f58::,AAFcvU7LIAoAABO-Xttw5A,7,WEB_ID,android,1


In [17]:
# save final file output
final_df.to_csv('cyber_1\\final_threshold_classified_users.csv', index=False)
print("Advanced classification with thresholds completed successfully!")

Advanced classification with thresholds completed successfully!


In [15]:
# in the first stage of the calssification we used a strict thresholds to clean up noise like: temporary logins, and idle IDs. 
# We defined that an ID would be considered a true 'additional device' only if it represented significant activity:
# at least X% (5%) of the user's total activity, and the duration of the impressions spread over more than 24 hours (one day).'
# ' Only IDs that met these two conditions moved to the **next stage**, where the overlap between them was checked(user number one was flagged as one device becausse of this check).
# 'This approach ensures protection against false positives of users who logged in once from a foreign device or reinstall OS app or upgraded to a new phone.
